# 02 Baselines

**Goal:** build the train, validation and test split from the EDA, set a rule based baseline, and fit a first logistic regression.

Decisions carried over from `01_eda.ipynb`:
- Split inside 1 to 10 Sep 2022, with the 11 to 18 Sep tail held apart.
- `self_transfer` is a feature. Payment format and payment currency are ablated.
- Metrics are PR-AUC and recall at a fixed false positive rate.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")

TX_COLS = [
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
]
TX_DTYPES = {
    "from_bank": "int32", "from_account": "category",
    "to_bank": "int32", "to_account": "category",
    "amount_received": "float64", "receiving_currency": "category",
    "amount_paid": "float64", "payment_currency": "category",
    "payment_format": "category", "is_laundering": "int8",
}


def load_transactions(path):
    tx = pd.read_csv(path, header=0, names=TX_COLS, dtype=TX_DTYPES)
    tx["timestamp"] = pd.to_datetime(tx["timestamp"], format="%Y/%m/%d %H:%M")
    tx["date"] = tx["timestamp"].dt.normalize()
    tx["self_transfer"] = (
        (tx["from_account"].astype(str) == tx["to_account"].astype(str))
        & (tx["from_bank"] == tx["to_bank"])
    )
    return tx


tx = load_transactions(DATA / "HI-Small_Trans.csv")
print(f"rows: {len(tx):,}")

rows: 5,078,345


In [2]:
SPLITS = {
    "train": ("2022-09-01", "2022-09-06"),
    "validation": ("2022-09-07", "2022-09-08"),
    "test": ("2022-09-09", "2022-09-10"),
}


def split_by_date(tx, splits=SPLITS):
    return {
        name: tx[(tx["date"] >= start) & (tx["date"] <= end)]
        for name, (start, end) in splits.items()
    }


parts = split_by_date(tx)
train, val, test = parts["train"], parts["validation"], parts["test"]

for name, part in parts.items():
    print(f"{name:<11} rows: {len(part):>9,}   laundering: {part['is_laundering'].sum():>5,}")

train       rows: 3,248,921   laundering: 2,530
validation  rows:   965,524   laundering: 1,036
test        rows:   862,792   laundering:   956


## Evaluation

Every model is scored the same way on the validation set:
- **PR-AUC** (average precision), the main metric at this class balance.
- **Recall at a fixed false positive rate** of 0.1% and 1%, which answers "how much laundering is caught if analysts can only review this many alerts".

A model that ranks at random has a PR-AUC equal to the laundering rate, about 0.001.

In [3]:
from sklearn.metrics import average_precision_score, roc_curve


def recall_at_fpr(y_true, scores, max_fpr):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return float(tpr[fpr <= max_fpr].max())


def evaluate(y_true, scores, name, fprs=(0.001, 0.01)):
    result = {"model": name, "pr_auc": average_precision_score(y_true, scores)}
    for f in fprs:
        result[f"recall_at_{f:.1%}_fpr"] = recall_at_fpr(y_true, scores, f)
    return result


results = []
y_val = val["is_laundering"].to_numpy()
print(f"validation laundering rate: {y_val.mean():.4%}")

validation laundering rate: 0.1073%


In [4]:
ach_flag = (val["payment_format"] == "ACH").to_numpy().astype(int)

tp = int(((ach_flag == 1) & (y_val == 1)).sum())
fp = int(((ach_flag == 1) & (y_val == 0)).sum())
fn = int(((ach_flag == 0) & (y_val == 1)).sum())
tn = int(((ach_flag == 0) & (y_val == 0)).sum())

print(f"alerts raised: {tp + fp:,} of {len(val):,} transactions")
print(f"precision: {tp / (tp + fp):.2%}")
print(f"recall: {tp / (tp + fn):.2%}")
print(f"false positive rate: {fp / (fp + tn):.2%}")

results.append(evaluate(y_val, ach_flag, "ACH rule"))
pd.DataFrame(results)

alerts raised: 116,910 of 965,524 transactions
precision: 0.76%
recall: 85.91%
false positive rate: 12.03%


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,ACH rule,0.006691,0.0,0.0
